# tutor

> The loop: pick, ask, let the learner struggle, grade, update, teach the smallest step.

This module runs the learning loop. `pick` chooses the concept and the depth, and a model writes the question.
The learner answers and can ask for hints. The model grades the answer, and the learner model updates the concept.
After a miss, the tutor teaches the smallest step. Each surface (the web app, the ramabana terminal agent, a notebook) uses `Tutor` and `Session`.

In [ ]:
#| default_exp tutor

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

Import the tools that this module uses, and the other abhyasa modules that the tutor calls.

In [ ]:
#| export
import json, random, time, uuid
from fastcore.utils import *
from abhyasa.store import Learning
from abhyasa.model import DEPTHS, KIND_AT, GOAL, update, pick, recall, outcome, next_depth, learned, weak, understanding
from abhyasa.llm import ask_json, for_task
from abhyasa.ground import VaultSource, ground, ground_text
from abhyasa.profile import PACES, profile_text, observe, calibration_text
from abhyasa.journey import current, name_gaps
from abhyasa.method import METHODS, EXAMPLES_TO, choose, methods_text, posterior, mean

## Tries and hints

A wrong answer does not end the question. The tutor names the first error, gives no fix, and the learner tries again.
After `MAX_TRIES` wrong tries, or when the learner stops, the tutor teaches the smallest step.
Then the state of the concept records the miss. A pass after a wrong try counts as `fragile`.
The best try counts: each try keeps its grade. When a try had `NEAR` or more, the close is a `fragile` pass, not a miss, because the learner had the idea.

Each open question keeps a thread, `q.turns`: the question, then each message of the learner and each reply of the tutor, in order.
A message of the learner is only what they say this time, often a follow-up to the last nudge. They do not send the full answer again.
The grade, the hints and the lesson read the whole thread.

`HINTS` is the hint ladder, from the least to the most revealing. The model gets one rung for each hint request.

In [ ]:
#| export
MAX_TRIES = 3
NEAR = 7      # a try graded this or more shows the idea; a later miss then counts as `fragile`
HINTS = ('Point their attention at the part of the problem that matters. Name nothing.',
         'Describe the pattern or shape of the solution, in general terms.',
         'Name the category of idea or method that applies.',
         'Name the specific idea, rule or theorem to use, without using it.',
         'Outline the move: the first step and why, but not its result.',
         'Do one worked step of the solution. Leave the rest to them.')

`DEPTH_ASK` tells the model what a question at each depth must make the learner do.

In [ ]:
#| export
DEPTH_ASK = {1: 'recognise it: a multiple-choice question where each wrong option matches one specific misconception',
             2: 'recall it: state the rule, definition or fact in their own words',
             3: 'explain it: say why it is true or why it works, in their own words',
             4: 'apply it: solve a concrete problem that needs it; not a restatement',
             5: 'transfer it: use it in a new situation, or connect it to a different idea or subject'}

`TUTOR_SP` is the system prompt for questions, hints and lessons.

In [ ]:
#| export
TUTOR_SP = """You are a tutor whose only goal is that the learner gets better at thinking. You never think for them.
Rules:
- Never give the answer, or a hint that names the answer, before the learner tries.
- Ask questions that need reasoning, not recitation. Prefer new cases over the textbook example.
- Keep it short and plain. Use LaTeX between $...$ for maths. Use the learner's subject terms correctly.
- The learner is an adult expert-learner; never be patronising, never praise effort, never pad.
- Write in ASD-STE100 Simplified Technical English at a US grade 9 reading level: short sentences (at most 20 words),
  active voice, one instruction per sentence, simple words. Technical terms of the subject are allowed."""

## Prompts

Each prompt function makes the text for one model call. The prompt text is an instruction to the model.

`_concept_block` describes the concept and the learner. `question_prompt` asks for one question at one depth. `who` is the profile of the learner as text (see `profile`), so the question uses their world.

In [ ]:
#| export
def _concept_block(c, ctx='', who=''):
    mis = '; '.join(c.get('misconceptions') or []) or 'none known'
    return (f"Concept: {c['name']} (subject: {c.get('subject') or 'general'})\nStatement: {c.get('statement') or '(none given)'}\n"
            f"Learner: floor {DEPTHS[c.get('floor', 0)]}, ceiling {DEPTHS[c['ceiling']] if c.get('ceiling', 6) < 6 else 'not found yet'}; "
            f"misconceptions seen: {mis}" + (f"\nFrom the learner's own vault (use only if relevant):\n{ctx}" if ctx else '')
            + (f"\nAbout the learner (set the question in their world: use the kinds of example that help them):\n{who}" if who else ''))

def _way(m): return f"The way to teach this time: {METHODS[m]}\n" if m in METHODS else ''

def question_prompt(c, depth, avoid=(), ctx='', who='', bridge=(), method=''):
    "The prompt that writes one question on `c` at `depth` in `method`; `bridge` names known ideas that a transfer question can connect to."
    kind = KIND_AT[depth]
    shape = ('{"question": str, "options": [{"text": str, "misconception": str}], "correct": int, "answer": str, "rubric": [str]}'
             if kind == 'choice' else '{"question": str, "answer": str, "rubric": [str]}')
    av = '\n'.join(f'- {q}' for q in avoid)
    return (f"{_concept_block(c, ctx, who)}\n\nWrite ONE question that makes the learner {DEPTH_ASK[depth]}.\n"
            f"Aim it just past what they reliably know, so they must struggle a little. Ask ONE thing, in under 60 words. Write only the final question: no drafts, no corrections, no 'no: take'.\n"
            + (f"Target this misconception if it fits: {c['misconceptions'][-1]}\n" if c.get('misconceptions') else '')
            + (f"Ideas that the learner knows well (connect to one only if the link is real): {'; '.join(bridge)}\n" if bridge else '') + _way(method)
            + (f"Do not repeat these earlier questions:\n{av}\n" if av else '')
            + f"`answer` is a model answer the learner will not see; `rubric` lists the 1-4 ideas that a correct answer must contain. "
            "Never make a rubric point a required word, a phrase or a step of a named method. The question can name a method; the idea counts without the name."
            + (" For options: 3 or 4, one correct (index in `correct`), each wrong one tied to a named misconception." if kind == 'choice' else '')
            + f"\nReturn JSON: {shape}")

A transfer question asks the learner to use an idea in a new place. The best new place is one that they already know. `bridges` names up to three concepts that the learner knows to depth 3 or more: from other subjects first, then linked ones, then the deepest. Thus a transfer question on attention can use a Sanskrit or jyotiṣa idea.

In [ ]:
#| export
def bridges(c, concepts, n:int=3) -> list:
    "The names of up to `n` concepts known to depth 3 or more, for a transfer question on `c`: other subjects first, then linked ones."
    links = {*(c.get('prereqs') or ()), *(c.get('related') or ())}
    ks = [x for x in concepts if x['slug'] != c['slug'] and x.get('floor', 0) >= 3]
    return [x['name'] for x in sorted(ks, key=lambda x: (x.get('subject') == c.get('subject'), x['slug'] not in links, -x.get('floor', 0)))[:n]]

`thread_text` writes the thread as plain lines for a prompt. `GRADE_SP` and `grade_prompt` grade the conversation against the
hidden model answer and the rubric. The latest message comes last, with the confidence that the learner gave it.

In [ ]:
#| export
def thread_text(q) -> str:
    "The conversation on `q` so far, one turn to a line, in order."
    def line(t):
        if t['who'] == 'tutor': return f"Tutor ({t.get('kind', 'reply')}): {t['text']}"
        return f"Learner: {t.get('text') or '(nothing)'}" + (f"\n  (think-aloud: {t['thinking']})" if t.get('thinking') else '')
    return '\n'.join(map(line, q.get('turns') or []))

GRADE_SP = """You grade one answer: what a learner says over a conversation on one question. The learner builds the answer over several messages.
A new message is often only a follow-up to your last nudge, not a full answer. Judge the reasoning in all their messages
together, including the think-aloud, not the wording.
- `correct`: true only when their messages together meet every rubric point in substance. A rubric point is met when a correct
  equation, formula or statement in their messages contains or implies it. They need not say each step in words.
  A right result from wrong reasoning is not correct.
- When the answer is right in substance and only the method or framing that the question names is missing, set `correct` true.
  Then use the deeper link in `feedback` to tie the answer to that method. Do not mark it wrong for missing words.
- `first_error`: where the reasoning still goes wrong now, said in one sentence, WITHOUT the fix. Empty when correct.
- `misconception`: the wrong mental model behind the error, in under 12 words, or empty.
- `feedback`: when correct, confirm in one line and add one deeper link or consequence. When wrong, reply to their latest
  message: say in a few words what in it is right, if anything, then ask one question that points at the next gap.
  Never repeat an earlier tutor turn. Never give the answer.
- When their latest message says they do not know a term or an idea ("I don't know what the loop means"), `feedback` first
  explains that term in one or two plain sentences, without the answer to the question. Then it asks its one question.
- Their messages often come from speech to text. Read past transcription noise: repeated words, a wrong homophone, a garbled
  term or formula, when the meaning is clear. When the latest message restates an earlier point, judge them together.
  Never mark them wrong for the transcription alone.
- `gaps`: each idea they say they do not know, as a short concept name in the subject ("weighted sum as a loop", not "the loop"); else [].
- `asks_for`: when they ask to be shown in a certain way ("show me with code", "with an equation"), one of code, maths, analogy,
  example, diagram, words; else "".
- `grade`: 0-10, as real partial credit. Give 7-10 when the idea is right. Give 4-6 when part of it is right. Give 0-3 when it is wrong or missing.
  When `correct` is false, `grade` is 6 or less.
- Write `feedback` and `first_error` in ASD-STE100 Simplified Technical English, grade 9: short sentences, active voice.
  Speak to the learner as "you"; never say "the learner"."""

def grade_prompt(q, answer, thinking='', confidence=None):
    "The prompt that grades the conversation on question `q`, with `answer` as the latest message of the learner."
    return (f"Concept: {q.name}\nDepth asked: {DEPTHS[q.depth]}\nQuestion: {q.question}\nModel answer (hidden from learner): {q.answer}\n"
            f"Rubric: {json.dumps(q.rubric)}\n\nThe conversation so far, in order:\n{thread_text(q) or 'Tutor (question): ' + q.question}\n\n"
            f"Their latest message follows; it may be only a follow-up to the last tutor turn.\n"
            f"Learner's answer: {answer or '(none)'}\nLearner's think-aloud: {thinking or '(none)'}\nLearner's confidence: {'not given' if confidence is None else f'{confidence}/100'}\n\n"
            + (f"Passages from the learner's own sources (judge against these too, where they apply):\n{q.ctx}\n\n" if q.get('ctx') else '')
            + (f"Write `feedback` in this way. {_way(q.get('method'))}" if q.get('method') in METHODS else '')
            + 'Return JSON: {"correct": bool, "grade": int, "first_error": str, "misconception": str, "feedback": str, "gaps": [str], "asks_for": str}')

`hint_prompt` asks for one rung of the hint ladder. It reads the thread, so the hint fits what the learner said.

In [ ]:
#| export
def hint_prompt(q, level):
    "The prompt for hint `level` (0-5) on question `q`."
    given = '\n'.join(f'- {h}' for h in q.hints)
    return (f"Concept: {q.name}\nQuestion: {q.question}\nModel answer (hidden): {q.answer}\n"
            + (f"The conversation so far:\n{thread_text(q)}\n" if q.get('turns') else f"Wrong tries so far: {json.dumps(q.tries)}\n")
            + (f"Hints already given:\n{given}\n" if given else '')
            + f"\nGive hint level {level + 1} of 6: {HINTS[level]}\n" + _way(q.get('method'))
            + "One or two sentences. Do not reveal the answer. Reply with the hint only.")

`teach_prompt` asks for the lesson after a miss. The lesson ends with a check question.

In [ ]:
#| export
def teach_prompt(q, first_error):
    "The prompt that teaches the smallest step after a miss."
    return (f"Concept: {q.name}\nStatement: {q.statement}\nQuestion: {q.question}\nModel answer: {q.answer}\n"
            + (f"The conversation:\n{thread_text(q)}\n" if q.get('turns') else f"Their tries: {json.dumps(q.tries)}\n")
            + f"Their first error: {first_error or 'they did not know'}\n\n"
            + (f"Passages from their own sources; build on one and name it, if it fits:\n{q.ctx}\n\n" if q.get('ctx') else '')
            + (f"About the learner (explain in their world, with the kinds of example that help them):\n{q.who}\n\n" if q.get('who') else '')
            + _way(q.get('method')) + "Teach the smallest step that fixes that first error, in under 120 words. Build on what they got right. "
            "Then show the model answer briefly. End with one line that starts 'Check:' and asks them to predict "
            "something new that the fixed idea implies (they will answer it next).")

## The tutor

`Tutor` keeps the open questions and the check questions from lessons.
`_ready_q` finds a question that came with the source of a concept.
After each `observe_every` closed questions, `observe` (from `profile`) reads the recent answers in the background and updates the profile.

In [ ]:
#| export
def _ready_q(c, depth, asked):
    "A ready question from the source of the concept at this depth, not asked before."
    qs = [q for q in c.get('questions') or () if q.get('depth', 2) == depth and q['q'] not in asked]
    return qs[0] if qs else None

#: functions `f(tutor, q, r)` that run after a question closes, in order; each can add to the result `r` (the demo module adds a demo)
CLOSE_HOOKS = []

class Tutor:
    "The learning loop over a `Learning` store with an `ask(prompt, sp)` model; `pool` runs background jobs."
    def __init__(self, learning:Learning, ask, goal:int=GOAL, context:bool=True, sources=None, pool=None, observe_every:int=6):
        store_attr()
        self.closed = 0    # questions closed; each `observe_every` closes, the profile learns from the answers
        if sources is None: self.sources = [VaultSource(learning)]
        self.open = {}     # question id -> question, while the learner works on it
        self.shut = {}     # question id -> question, for the last 20 closed questions, so you can ask for a lesson later
        self.checks = {}   # concept -> (the check question of the lesson, the lesson), asked next on that concept
        self.jobs = {}     # concept -> a background job, for example a demo that a model makes
        self.rng = random.Random()   # the method draws; a test sets a seed

    def _ground(self, c) -> L:
        "Up to three passages on concept `c` from the sources of the learner, so a question can use what they read and did."
        if not (self.context and self.sources): return L()
        terms = [c['name'], *(c.get('aliases') or [])]
        try: return ground(' '.join(terms[:4]), self.sources, 3, must=terms)
        except Exception: return L()

## Ask a question

`ask_on` uses the first source that it finds: the check from the last lesson, a ready question, or a new question from the model.
It shuffles the options of a choice question, and starts the thread with the question.
`public` removes the fields that the learner must not see.

In [ ]:
#| export
@patch
def ask_on(self:Tutor, c, depth:int, why:str='', session:str='', mode:str=None) -> AttrDict:
    "Write a question on concept `c` at `depth` in a method from `choose`, keep it open, and return it without its answer fields. `mode='drill'` uses your best method."
    asked = self.learning.attempts(c['slug'], limit=20).attrgot('question')
    q = AttrDict(id=uuid.uuid4().hex[:10], concept=c['slug'], name=c['name'], statement=c.get('statement', ''),
                 subject=c.get('subject', ''), floor=c.get('floor', 0), depth=depth, kind=KIND_AT[depth], why=why, session=session,
                 hints=[], tries=[], at=time.time())
    ps, p, st = self._ground(c), self.learning.profile(), self.learning.stats()
    q.method, note = choose(st, c.get('subject', ''), mode == 'drill' or weak(c), self.rng, p.get('explore', 'sometimes'),
                            [EXAMPLES_TO[e] for e in p.examples if e in EXAMPLES_TO])
    if note: q.why = f'{why}; {note}'.lstrip('; ')
    q.who = '\n'.join(x for x in (profile_text(p), calibration_text(AttrDict(st.calib)) if st.calib else '', methods_text(st, c.get('subject', ''))) if x)
    q.update(ctx=ground_text(ps), grounds=[dict(src=p.src, ref=p.ref, title=p.title, url=p.url) for p in ps])
    if (ck := self.checks.pop(c['slug'], None)):   # the check of a lesson counts for the method of the lesson
        q.update(question=ck[0], answer=ck[1], rubric=['the prediction follows from the lesson'], ready=True, method=ck[2], why=f'{why}; the check from the lesson'.lstrip('; '))
    elif (r := _ready_q(c, depth, asked)): q.update(question=r['q'], answer=r.get('a', ''), rubric=r.get('rubric') or [r.get('a', '')], ready=True)
    else:
        r = ask_json(for_task(self.ask, 'question'), question_prompt(c, depth, asked[:8], q.ctx, q.who, bridges(c, self.learning.concepts()) if depth == 5 else (), q.method), TUTOR_SP)
        q.update(question=r['question'], answer=r.get('answer', ''), rubric=r.get('rubric', []))
        if q.kind == 'choice' and r.get('options'):
            opts = list(enumerate(r['options']))
            random.shuffle(opts)
            q.options = [o for _, o in opts]
            q.correct = next(i for i, (j, _) in enumerate(opts) if j == r.get('correct', 0))
        elif q.kind == 'choice': q.kind = 'recall'
    q.turns = [dict(who='tutor', kind='question', text=q.question)]
    self.open[q.id] = q
    return public(q)

def public(q) -> AttrDict:
    "A question as the learner can see it: no model answer, no rubric, no correct option, no misconception tags."
    out = AttrDict({k: v for k, v in q.items() if k not in ('answer', 'rubric', 'correct', 'options', 'statement', 'ctx', 'grounds', 'who')})
    if q.get('options'): out.options = [o['text'] for o in q.options]
    return out

`next` lets `pick` choose the concept, then asks a question on it. `paced` moves the depth for the pace in the profile of the learner.

In [ ]:
#| export
def paced(c, depth:int, pace:str='steady') -> int:
    "The depth for the pace of the learner: at `gentle`, at most one above the floor; at `fast`, one more than `pick` gives."
    if c.get('shaky'): return depth
    if pace == 'gentle': return max(1, min(depth, c.get('floor', 0) + 1))
    if pace == 'fast': return min(5, depth + 1)
    return depth

@patch
def next(self:Tutor, focus:str=None, subject:str=None, session:str='', skip=(), near=(), pace:str=None, mode:str=None) -> AttrDict|None:
    "The next question, chosen by `pick` at `pace` (by default the pace in the profile), or None when there is nothing to learn or review."
    r = pick(self.learning.concepts(), focus=focus, subject=subject, goal=self.goal, skip=skip, near=near)
    if not r: return None
    c, d, why = r
    return self.ask_on(c, paced(c, d, pace or self.learning.profile().pace), why, session=session, mode=mode)

`flow` keeps a session in its band, at about 85% success. It reads the last outcomes of the session. After two misses in a row, the next question is one pace easier. After three solid passes in a row, it is one pace harder. The note says why, so the learner sees the change.

In [ ]:
#| export
MISSES = ('gap', 'misconception')

def flow(log, pace:str='steady', n:int=3) -> tuple:
    "The `(pace, note)` for the next question of a session: one pace easier after two misses, one pace harder after `n` solid passes."
    ps, last = list(PACES), [r.get('outcome') for r in log[-n:]]
    i = ps.index(pace) if pace in ps else 1
    if len(last) > 1 and set(last[-2:]) <= set(MISSES): i, note = max(i - 1, 0), 'easier: you missed the last two'
    elif len(last) == n and set(last) == {'solid'}: i, note = min(i + 1, len(ps) - 1), f'harder: {n} clean passes in a row'
    else: note = ''
    return ps[i], note if ps[i] != pace else ''

## Hints

Each call to `hint` gives the next rung of the ladder. After the last rung, it gives the last hint again.

In [ ]:
#| export
@patch
def hint(self:Tutor, qid:str) -> str:
    "The next hint on the ladder, from the least to the most revealing."
    q = self.open[qid]
    if len(q.hints) >= len(HINTS): return q.hints[-1]
    h = for_task(self.ask, 'hint')(hint_prompt(q, len(q.hints)), TUTOR_SP).strip()
    q.hints.append(h)
    q.setdefault('turns', []).append(dict(who='tutor', kind='hint', text=h))
    return h

## Grade an answer

`_grade` grades a choice question in code and other questions with the model.

In [ ]:
#| export
@patch
def _grade(self:Tutor, q, answer, thinking, confidence):
    if q.kind == 'choice' and q.get('options'):
        try: i = int(answer)
        except (TypeError, ValueError): i = next((j for j, o in enumerate(q.options) if o['text'] == answer), -1)
        ok = i == q.correct
        mis = '' if ok or not (0 <= i < len(q.options)) else q.options[i].get('misconception', '')
        return AttrDict(correct=ok, grade=10 if ok else 0, misconception=mis, first_error=mis and f'This option assumes: {mis}',
                        feedback='Correct.' if ok else 'Not this one either. Which option is left?' if q.tries else 'Not this one. Look again at what each option assumes.')
    return AttrDict(ask_json(for_task(self.ask, 'grade'), grade_prompt(q, answer, thinking, confidence), GRADE_SP))

`answer` adds the message of the learner to the thread, and the nudge after a wrong try. A pass or the last try closes the question.

In [ ]:
#| export
@patch
def _heard(self:Tutor, q, g):
    "Act on what the grader heard: keep each idea you say you do not know as a gap, and switch to the way you ask for."
    if (gs := [x for x in g.get('gaps') or [] if isinstance(x, str) and x.strip()]):
        q.gaps = [*q.get('gaps', []), *gs]
        job = lambda: name_gaps(self.learning, for_task(self.ask, 'extract'), q.concept, gs, self.sources)
        if self.pool: self.pool.submit(job)
        else:
            try: job()
            except Exception: pass
    if g.get('asks_for') in METHODS: q.method = g['asks_for']

@patch
def answer(self:Tutor, qid:str, answer:str='', thinking:str='', confidence:int=None, dont_know:bool=False) -> AttrDict:
    "Grade an answer. A wrong try keeps the question open with a nudge. A pass or the last try closes it and updates the concept."
    q = self.open[qid]
    if dont_know: return self.give_up(qid, confidence=confidence, dont_know=True)
    g = self._grade(q, answer, thinking, confidence)
    self._heard(q, g)
    said = q.options[int(answer)]['text'] if q.get('options') and str(answer).isdigit() and int(answer) < len(q.options) else answer
    q.setdefault('turns', []).append(dict(who='you', text=said, thinking=thinking, confidence=confidence))
    if not g.get('correct') and len(q.tries) + 1 < MAX_TRIES:
        q.tries.append(dict(answer=answer, thinking=thinking, confidence=confidence, first_error=g.get('first_error', ''),
                            misconception=g.get('misconception', ''), grade=g.get('grade')))
        self.learning.record(q.concept, q.question, answer, thinking, False, confidence, q.depth, q.kind, len(q.hints),
                             grade=g.get('grade'), feedback=g.get('feedback', ''), first_error=g.get('first_error', ''),
                             misconception=g.get('misconception', ''), session=q.session, final=False, method=q.get('method', ''), gaps=g.get('gaps') or [])
        q.turns.append(dict(who='tutor', kind='nudge', text=g.get('feedback', ''), first_error=g.get('first_error', '')))
        return AttrDict(done=False, correct=False, feedback=g.get('feedback', ''), first_error=g.get('first_error', ''),
                        tries_left=MAX_TRIES - len(q.tries))
    if not g.get('correct'):
        q.tries.append(dict(answer=answer, thinking=thinking, confidence=confidence, first_error=g.get('first_error', ''), grade=g.get('grade')))
        return self._close(q, False, q.tries[0]['confidence'], g, teach=True)
    return self._close(q, True, confidence, g, answer=answer, thinking=thinking)

## Stop and teach

`give_up` closes the question as a miss, unless an earlier try was graded `NEAR` or more. `_close` updates the concept, teaches after a miss and records the attempt.
Then each function in `CLOSE_HOOKS` can add to the result. The demo module uses this hook to attach a demo after a miss.
The confidence of the first try counts, because the first try shows what the learner believed.

In [ ]:
#| export
@patch
def give_up(self:Tutor, qid:str, confidence:int=0, dont_know:bool=False) -> AttrDict:
    "Close the question as a miss and teach the smallest step. A shown answer is a miss, unless a try had a grade of `NEAR` or more."
    q = self.open[qid]
    first = q.tries[0] if q.tries else {}
    g = AttrDict(correct=False, grade=0, first_error=first.get('first_error', ''), misconception=first.get('misconception', ''),
                 feedback='')
    return self._close(q, False, first.get('confidence', confidence), g, teach=True, dont_know=dont_know and not q.tries)

`_lesson` writes the lesson after a miss. When the model fails, the lesson is a note with the model answer.

In [ ]:
#| export
@patch
def _lesson(self:Tutor, q, first_error:str='') -> str:
    "A short lesson on `q` from its first error. When the model fails, a note with the model answer."
    try: lesson = for_task(self.ask, 'lesson')(teach_prompt(q, first_error), TUTOR_SP).strip()
    except Exception as e: return f"The tutor could not write the lesson ({type(e).__name__}). The model answer is: {q.answer or 'not given'}"
    if 'Check:' in lesson: self.checks[q.concept] = (lesson.rsplit('Check:', 1)[1].strip(), lesson, q.get('method', ''))
    return lesson

@patch
def _close(self:Tutor, q, correct, confidence, g, teach=False, answer='', thinking='', dont_know=False):
    # the lesson comes first: if the model fails here, nothing is saved yet, so a retry cannot count the same miss twice
    lesson = self._lesson(q, g.get('first_error', '')) if teach else ''
    best = max((t.get('grade') or 0 for t in q.tries), default=0)
    if not correct and not dont_know and best >= NEAR: correct, g = True, AttrDict({**g, 'grade': best, 'feedback': '', 'first_error': '', 'misconception': ''})   # the best try counts: a shaky pass
    c = self.learning.concept(q.concept)
    hints = len(q.hints) + 2 * bool(q.tries)   # a pass after a wrong try is never a clean pass
    u = update(c, q.depth, correct, confidence, hints, dont_know, g.get('misconception', ''))
    kind = u.pop('kind')
    self.learning.update(q.concept, **u)
    q.setdefault('turns', []).append(dict(who='tutor', kind='close', text=g.get('feedback', ''), lesson=lesson, outcome=kind))
    q.aid = self.learning.record(q.concept, q.question, answer or (q.tries[-1]['answer'] if q.tries else ''), thinking, correct, confidence,
                         q.depth, q.kind, hints, dont_know, g.get('grade'), g.get('feedback', '') or lesson,
                         g.get('first_error', ''), g.get('misconception', ''), session=q.session, final=True, outcome=kind, thread=q.turns,
                         method=q.get('method', ''), gaps=q.get('gaps', [])).id
    if q.get('method') and not dont_know: self.learning.credit_method(q.method, q.subject, kind)   # "I don't know" used no method
    del self.open[q.id]
    self.shut[q.id] = q
    if len(self.shut) > 20: del self.shut[next(iter(self.shut))]
    self.closed += 1
    if self.observe_every and self.pool and self.closed % self.observe_every == 0: self.pool.submit(self._observe)
    r = AttrDict(done=True, correct=correct, outcome=kind, feedback=g.get('feedback', ''), first_error=g.get('first_error', ''),
                 lesson=lesson, answer=q.answer if not correct else '', misconception=g.get('misconception', ''), floor=u.floor, ceiling=u.ceiling,
                 due=u.due, depth=q.depth, grounds=q.get('grounds', []), concept=q.concept, session=q.session)
    for f in CLOSE_HOOKS: r = f(self, q, r) or r
    return r

@patch
def _observe(self:Tutor):
    "Update the profile from the recent answers; a failure changes nothing."
    try: return observe(self.learning, for_task(self.ask, 'extract'))
    except Exception: return []

The tutor keeps the last 20 closed questions in `shut`. `teach` gives a lesson on one of them when you ask for it, for example after a shaky pass. It does not change the learner state. A second call gives the same lesson.

In [ ]:
#| export
SHAKY = 'They passed, but only after nudges or hints. Make the idea solid.'

@patch
def teach(self:Tutor, qid:str, another:bool=False) -> str:
    "A lesson on request for the closed question `qid`, kept in its thread and on its attempt. `another` uses a different method. The learner state does not change."
    q = self.shut[qid]
    done = [t for t in q.get('turns') or () if t.get('kind') == 'lesson']
    if done and not another: return done[-1]['text']
    if another:
        used = {q.get('method'), *(t.get('method') for t in done)}
        left = [m for m in METHODS if m not in used] or ['words']
        st = self.learning.stats()
        q.method = max(left, key=lambda m: mean(posterior(st, m, q.subject)))
    lesson = self._lesson(q, (q.tries[0].get('first_error') if q.tries else '') or SHAKY)
    q.setdefault('turns', []).append(dict(who='tutor', kind='lesson', text=lesson, method=q.get('method', '')))
    if q.get('aid'): self.learning.mark_attempt(q.aid, taught=len(done) + 1, teach_method=q.get('method', ''))
    return lesson

## A session

`Session` keeps one sitting together. Its id goes on each attempt, and it stops after `limit` questions (by default, the
session length in the profile of the learner). It asks about one concept for at most `run` questions in a row.
This lets the range between floor and ceiling close, and it keeps the learner interested.

A session is saved on the shelf after each move: when a question opens, after each answer and hint, and at the end. It keeps the log
of each closed question (with the floor before and after, and the thread) and the open question with its thread. Thus `Session.load`
continues it after a restart, with the same question, graded the same way.

A session in a journey asks about the current step of the journey. When the step is learned, the next question is on the next step.

In [ ]:
#| export
class Session:
    "One sitting: questions and answers, saved on the shelf after each move, so that a restart can continue it."
    def __init__(self, tutor:Tutor, focus:str=None, subject:str=None, limit:int=None, run:int=3, journey:str=None, sid:str=None, drill:bool=False):
        store_attr('tutor,focus,subject,run,journey,drill')
        self.limit = limit or tutor.learning.profile().session_len
        self.id, self.start, self.log, self.q, self.ended, self.near = sid or uuid.uuid4().hex[:10], time.time(), [], None, None, set()
        if journey and (j := tutor.learning.journey(journey)) is not None: self.near = {s['slug'] for s in j.steps}
        elif focus and (c := tutor.learning.concept(focus)) is not None:
            # a session on one concept stays in its subject, and goes next to its neighbours: links in either direction
            self.subject = subject or c.get('subject') or None
            self.near = {*c.get('prereqs', ()), *c.get('related', ())} | {x.slug for x in tutor.learning.concepts(self.subject) if focus in (*(x.get('prereqs') or ()), *(x.get('related') or ()))}

    def _skip(self):
        last = [r['concept'] for r in self.log[-self.run:]]
        return {last[0]} if len(last) == self.run and len(set(last)) == 1 else set()

    def next(self) -> AttrDict|None:
        "The next question, or None when the session is over (or its journey is done)."
        if self.ended or len(self.log) >= self.limit: return None
        skip, (pace, note) = self._skip(), flow(self.log, self.tutor.learning.profile().pace)
        mode = 'drill' if note.startswith('easier') else None   # after two misses: your best way, not a new one
        if (q := self._retest(pace) or self._drill(pace)): self.q = q
        elif self.journey:
            j = self.tutor.learning.journey(self.journey)
            if not (f := current(self.tutor.learning, j, skip, self.tutor.goal) if j else None): return None
            self.q = self.tutor.next(focus=f, session=self.id, skip=skip, near=self.near, pace=pace, mode=mode)
            if self.q and self.q.why.startswith('you chose'):   # the learner chose the journey, not the concept: say so
                self.q.why = self.tutor.open[self.q.id].why = self.q.why.replace(f'you chose {self.q.name}', f'the next step of your journey "{j.title}"', 1)
        else: self.q = self.tutor.next(focus=self.focus if self.focus not in skip else None, subject=self.subject, session=self.id, skip=skip, near=self.near, pace=pace, mode=mode)
        if self.q and note: self.q.why = self.tutor.open[self.q.id].why = f'{self.q.why}; {note}'
        self.save()
        return self.q

A check right after a lesson shows that you understood it. It does not show that you will remember it, because the lesson is still in your mind. Thus each concept that you miss in a session comes back one more time before the end. At least two other questions come between the miss and the retest. `retests` finds these concepts. A pass right after the miss does not count.

In [ ]:
#| export
def retests(log, gap:int=2) -> dict:
    "The concepts missed in a session log, with no pass at least `gap` questions after the miss: concept -> its last miss, oldest first."
    miss = {}
    for i, r in enumerate(log):
        k = r['concept']
        if r.get('outcome') in MISSES: miss.pop(k, None); miss[k] = i
        elif k in miss and i - miss[k] > gap: del miss[k]
    return miss

`_retest` asks one of them when only enough questions are left to retest each miss once.

In [ ]:
#| export
@patch
def _retest(self:Session, pace:str='steady', gap:int=2) -> AttrDict|None:
    "A retest of a miss from this session, when the questions left are only enough for the retests; else None."
    rt, n = retests(self.log, gap), len(self.log)
    if not rt or self.limit - n > len(rt): return None
    if not (k := first(k for k, i in rt.items() if n - 1 - i >= gap)) or (c := self.tutor.learning.concept(k)) is None: return None
    d = c.get('floor', 0) if learned(c, self.tutor.goal) else next_depth(c)
    return self.tutor.ask_on(c, paced(c, d, pace), 'a retest: you missed this earlier in this session', session=self.id, mode='drill')

A drill session ("Practice my weak spots") asks your weakest concepts first, each one time, with your best method. When none is left, the session goes on as usual.

In [ ]:
#| export
@patch
def _drill(self:Session, pace:str='steady') -> AttrDict|None:
    "In a drill session, the weakest concept not asked yet in this session, with your best method; else None."
    if not self.drill: return None
    asked = {r['concept'] for r in self.log}
    us = {c.slug: (c, understanding(c)) for c in self.tutor.learning.concepts(self.subject) if c.slug not in asked}
    if not (ws := sorted((x for x in us.values() if x[1].weak), key=lambda x: x[1].score)): return None
    c, u = ws[0]
    d = c.get('floor', 0) if learned(c, self.tutor.goal) else next_depth(c)
    return self.tutor.ask_on(c, paced(c, d, pace), f"a weak spot: {u.reasons[0] if u.reasons else 'you missed it'}", session=self.id, mode='drill')

`answer`, `give_up` and `hint` act on the open question and save the session. A closed question goes in the log with what moved.

In [ ]:
#| export
@patch
def _logged(self:Session, r, full):
    q = self.q
    self.log.append(dict(qid=q.id, concept=q.concept, name=q.name, depth=q.depth, why=q.get('why', ''), question=q.question, method=q.get('method', ''),
                         correct=r.correct, outcome=r.outcome, floor0=q.get('floor', 0), floor=r.floor, misconception=r.get('misconception', ''),
                         turns=list((full or {}).get('turns') or []), at=time.time()))

@patch
def answer(self:Session, *args, **kw) -> AttrDict:
    full = self.tutor.open.get(self.q.id)
    r = self.tutor.answer(self.q.id, *args, **kw)
    if r.done: self._logged(r, full)
    self.save()
    return r

@patch
def give_up(self:Session, **kw) -> AttrDict:
    full = self.tutor.open.get(self.q.id)
    r = self.tutor.give_up(self.q.id, **kw)
    self._logged(r, full)
    self.save()
    return r

@patch
def hint(self:Session) -> str:
    h = self.tutor.hint(self.q.id)
    self.save()
    return h

`save` writes the session to the shelf. `summary` gives one line. `end` closes the session. `load` reads a session back, with its open question.

In [ ]:
#| export
@patch
def summary(self:Session) -> str:
    "A short summary, and the text of the session document."
    n, ok = len(self.log), sum(r['correct'] for r in self.log)
    return f"{ok} of {n} passed. Concepts: {', '.join(sorted({r['concept'] for r in self.log})) or 'none'}."

@patch
def save(self:Session):
    "File the session on the shelf, with its log and its open question. A session with nothing in it is not filed."
    if not (self.log or self.q): return
    full = self.tutor.open.get(self.q.id) if self.q and not self.ended else None
    self.tutor.learning.log_session(self.id, self.summary(), start=self.start, updated=time.time(), focus=self.focus, subject=self.subject,
                                    journey=self.journey, drill=self.drill, limit=self.limit, log=self.log, open=dict(full) if full else None, ended=self.ended,
                                    asked=len(self.log), passed=sum(r['correct'] for r in self.log), concepts=sorted({r['concept'] for r in self.log}))

@patch
def end(self:Session) -> str:
    "Close the session and file it. Return its summary."
    self.ended = time.time()
    if self.q: self.tutor.open.pop(self.q.id, None)
    self.save()
    return self.summary()

@patch(cls_method=True)
def load(cls:Session, tutor:Tutor, sid:str):
    "The session `sid` from the shelf, with its open question open again; None for an unknown id or an old summary."
    m = tutor.learning.session(sid)
    if m is None or 'log' not in m: return None
    s = cls(tutor, m.get('focus'), m.get('subject'), m.get('limit'), journey=m.get('journey'), sid=sid, drill=bool(m.get('drill')))
    s.start, s.log, s.ended = m.get('start') or s.start, list(m.get('log') or []), m.get('ended')
    if (o := m.get('open')) and not s.ended:
        q = AttrDict(o)
        tutor.open[q.id] = q
        s.q = public(q)
    return s

`Session.teach` asks for that lesson on a question of the session, and keeps it in the log, so the thread shows it after a restart.

In [ ]:
#| export
@patch
def teach(self:Session, qid:str, another:bool=False) -> str:
    "A lesson on request for `qid`, a closed question of this session; the log keeps it."
    lesson = self.tutor.teach(qid, another)
    for r in self.log:
        if r.get('qid') == qid: r['turns'] = list(self.tutor.shut[qid].turns)
    self.save()
    return lesson

`moved` says what a session changed: for each concept, its floor before the first question and after the last one.

In [ ]:
#| export
def moved(log) -> list:
    "For each concept in a session log, in order: its name, the floor before and after, and the misconceptions found."
    out = {}
    for r in log:
        m = out.setdefault(r['concept'], dict(concept=r['concept'], name=r.get('name') or r['concept'], before=r.get('floor0', 0), after=r.get('floor0', 0),
                                              asked=0, passed=0, misconceptions=[]))
        m['after'], m['asked'], m['passed'] = r.get('floor', m['after']), m['asked'] + 1, m['passed'] + bool(r.get('correct'))
        if r.get('misconception') and r['misconception'] not in m['misconceptions']: m['misconceptions'].append(r['misconception'])
    return list(out.values())

## Tests

A fake model plays the tutor. Its grader passes an answer only when the answer has the words "rank position".
Two concepts are on the shelf, and `Reciprocal rank fusion` needs `Ranking`.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake

lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml', order=0)
lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml', prereqs=['ranking'], order=1)
def grader(p, sp):
    ok = 'rank position' in p.split("Learner's answer:")[1]
    return dict(correct=ok, grade=8 if ok else 3, first_error='You add raw scores.',
                               misconception='scores are comparable across systems', feedback='Look at what each list really shares.')
fake = Fake({'You grade one answer': grader, 'Give hint level': 'Look at what the two lists have in common.',
             'Teach the smallest step': 'Only positions are shared. Check: what happens with k = 0?',
             'Write ONE question': dict(question='Why does RRF use ranks, not scores?', answer='Scores are not comparable.', rubric=['ranks are comparable'])})
t = Tutor(lr, fake, context=False)

The first question is on the prereq, at depth 3. The learner does not see the answer or the rubric.

In [ ]:
q = t.next()
test_eq((q.concept, q.depth, q.kind), ('ranking', 3, 'explain'))   # rrf waits for its prereq
assert 'answer' not in q and 'rubric' not in q

A wrong answer keeps the question open and names the first error. Then a hint comes from the ladder.

In [ ]:
r = t.answer(q.id, 'sum the scores', confidence=90)
test_eq((r.done, r.correct, r.tries_left), (False, False, 2))
test_eq(r.first_error, 'You add raw scores.')
test_eq(t.hint(q.id), 'Look at what the two lists have in common.')

A pass after a wrong try and a hint is `fragile`. The floor stays, and the concept is `shaky` at depth 3.

In [ ]:
r = t.answer(q.id, 'use the rank position', thinking='positions are shared', confidence=90)
test_eq((r.done, r.correct, r.outcome), (True, True, 'fragile'))   # passed, but after a wrong try and a hint
c = lr.concept('ranking')
test_eq((c.floor, c.shaky, c.seen), (0, 3, 1))
test_eq(len(lr.attempts('ranking')), 2)

The thread holds each turn in order: the question, the wrong try, its nudge, the hint and the pass.

In [ ]:
test_eq([(x['who'], x.get('kind')) for x in q.turns],
        [('tutor', 'question'), ('you', None), ('tutor', 'nudge'), ('tutor', 'hint'), ('you', None), ('tutor', 'close')])
test_eq(lr.attempts('ranking')[0].thread[1]['text'], 'sum the scores')   # the final attempt keeps the thread

Three wrong tries close the question, keep the misconception and teach.

In [ ]:
q = t.next(focus='ranking')
test_eq(q.depth, 3)    # shaky at explain: ask at explain again
for _ in range(3): r = t.answer(q.id, 'sum the scores', confidence=85)
test_eq((r.done, r.outcome), (True, 'misconception'))
assert 'Check:' in r.lesson
c = lr.concept('ranking')
test_eq((c.ceiling, c.misconceptions), (3, ['scores are comparable across systems']))

When the lesson model fails, the question still closes, the miss counts once, and the lesson says what happened.

In [ ]:
fl = temp_learning(); fl.put('Ranking', 'Order items by a score.', 'ml')
def broken(p, sp=''):
    if 'Teach the smallest step' in p: raise RuntimeError('the model is down')
    return fake(p, sp)
ft = Tutor(fl, broken, context=False)
q = ft.next()
for a in ('sum the scores', 'add the scores', 'average the scores'): r = ft.answer(q.id, a, confidence=85)
test_eq((r.done, q.id in ft.open), (True, False))
assert 'could not write the lesson' in r.lesson and 'Scores are not comparable' in r.lesson
c = fl.concept('ranking')
test_eq((c.seen, c.misses), (1, 1))
test_eq(len([a for a in fl.attempts('ranking') if a.get('final')]), 1)

The check question from the lesson comes next.

In [ ]:
q = t.next(focus='ranking')
test_eq(q.question, 'what happens with k = 0?')   # the lesson's check comes next
t.open.clear()

A ready question from a source comes before a question from the model.

In [ ]:
lr.put('Sandhi', 'Sounds change where words meet.', 'sanskrit', questions=[dict(q='Join na + asti.', a='nāsti', depth=3)])
q = t.next(subject='sanskrit')
test_eq(q.question, 'Join na + asti.')

Code grades a choice question. A wrong option gives the misconception that it matches.

In [ ]:
fake.rules['Write ONE question'] = dict(question='Which is a sandhi?', options=[dict(text='na + asti → nāsti', misconception=''),
    dict(text='na + asti → naasti', misconception='vowels just sit side by side')], correct=0, answer='', rubric=[])
t.open.clear()
lr.update('sandhi', floor=0, ceiling=2)
q = t.next(subject='sanskrit')
test_eq(q.kind, 'choice')
r = t.answer(q.id, q.options.index('na + asti → naasti'), confidence=90)
test_eq(r.first_error, 'This option assumes: vowels just sit side by side')

A session stops after `limit` questions and files a summary.

In [ ]:
fake2 = Fake({k: fake.rules[k] for k in ('You grade one answer', 'Teach the smallest step')},
             dflt=dict(question='What does a rank tell you?', answer='Its position.', rubric=['position']))
s = Session(Tutor(temp_learning(), fake2, context=False), limit=2)
s.tutor.learning.put('Ranking', 'Order items by a score.', 'ml')
assert s.next() is not None
s.give_up()
assert s.next() is not None
test_eq(s.answer('the rank position', confidence=80).correct, True)
test_eq(s.next(), None)
test_eq(s.summary(), '1 of 2 passed. Concepts: ranking.')
test_eq(s.tutor.learning.session(s.id).asked, 2)          # filed as it goes, not only at the end

A session on one concept stays in its subject. It asks about the neighbors of the concept first.

In [ ]:
sl = temp_learning()
sl.put('Attention', 'Weights from query-key dot products.', 'ml', order=0)
sl.put('Softmax', 'Turns scores into weights that sum to 1.', 'ml', related=['attention'], order=5)
sl.put('Sandhi', 'Sounds change where words meet.', 'sanskrit', order=0)
ss = Session(Tutor(sl, fake2, context=False), focus='attention')
test_eq((ss.subject, ss.near), ('ml', {'softmax'}))
sl.update('attention', floor=4, seen=1, due=1e12)   # learned, and not due: the session moves on, and stays in ml
test_eq(ss.next().concept, 'softmax')

A session is saved with its open question. `load` continues it in a new tutor, as after a restart, and grades the same question.

In [ ]:
pl = temp_learning(); pl.put('Ranking', 'Order items by a score.', 'ml')
ps = Session(Tutor(pl, fake2, context=False), limit=3)
q1 = ps.next(); ps.give_up()
q2 = ps.next(); ps.answer('a score', confidence=60)       # a wrong try: the question stays open
t2 = Tutor(pl, fake2, context=False)                      # a restart: a new tutor with nothing open
back = Session.load(t2, ps.id)
test_eq((back.q.id, back.q.question, len(back.log), back.limit), (q2.id, q2.question, 1, 3))
assert 'answer' not in back.q and back.q.id in t2.open    # the learner view, and the full question in the tutor
test_eq(back.answer('the rank position', confidence=80).correct, True)
test_eq([(r['outcome'], r['floor0']) for r in back.log], [('gap', 0), ('fragile', 0)])
test_eq([t['who'] for t in back.log[-1]['turns']], ['tutor', 'you', 'tutor', 'you', 'tutor'])

`moved` gives the floor before and after for each concept of the log.

In [ ]:
test_eq([(m['concept'], m['before'], m['after'], m['asked'], m['passed']) for m in moved(back.log)], [('ranking', 0, 0, 2, 1)])
test_eq(moved([dict(concept='a', name='A', floor0=2, floor=3, correct=True), dict(concept='a', floor0=3, floor=4, correct=True, misconception='x')]),
        [dict(concept='a', name='A', before=2, after=4, asked=2, passed=2, misconceptions=['x'])])

`end` closes the session. An ended session loads with no open question and gives no more questions.

In [ ]:
back.next(); back.end()
again = Session.load(Tutor(pl, fake2, context=False), ps.id)
test_eq((again.q, again.next(), bool(again.ended)), (None, None, True))
test_is(Session.load(t2, 'nope'), None)

A session in a journey asks about the current step. When the step is learned, it goes to the next step. It stops at the end of the journey.

In [ ]:
jl = temp_learning()
for n in ('Tokenizers', 'Embeddings', 'Sandhi'): jl.put(n, f'{n} statement.', 'ml' if n != 'Sandhi' else 'sanskrit')
jl.update('sandhi', seen=1, due=1.)                     # a review due elsewhere does not break in
j = jl.put_journey(dict(id='llms', title='LLMs', goal='LLMs', steps=[dict(slug='tokenizers', status='todo'), dict(slug='embeddings', status='todo')]))
js = Session(Tutor(jl, fake2, context=False), journey='llms')
q = js.next()
test_eq((q.concept, q.why.split(';')[0]), ('tokenizers', 'the next step of your journey "LLMs"'))   # a method note can follow
js.give_up(); jl.update('tokenizers', floor=4)
test_eq(js.next().concept, 'embeddings')
js.give_up(); jl.update('embeddings', floor=4)
test_eq((js.next(), jl.session(js.id).journey), (None, 'llms'))

The profile goes into the question prompt. At the `gentle` pace a new concept starts at depth 1, not 3.

In [ ]:
gl2 = temp_learning(); gl2.put('Ranking', 'Order items by a score.', 'ml')
gl2.set_profile(background='codes in Python', pace='gentle')
fk = Fake(dflt=dict(question='Which keeps its scale?', options=[dict(text='a rank', misconception=''), dict(text='a score', misconception='x')], correct=0, answer='', rubric=[]))
q = Tutor(gl2, fk, context=False).next()
test_eq(q.depth, 1)
assert 'codes in Python' in fk.calls[-1][0] and 'who' not in q

`flow` moves the pace by one step, and only from the last outcomes. A note comes only when the pace changes.

In [ ]:
sol, gap = dict(outcome='solid'), dict(outcome='gap')
test_eq(flow([]), ('steady', ''))
test_eq(flow([sol, gap, dict(outcome='misconception')]), ('gentle', 'easier: you missed the last two'))
test_eq(flow([sol] * 3, 'gentle'), ('steady', 'harder: 3 clean passes in a row'))
test_eq(flow([gap, gap], 'gentle'), ('gentle', ''))     # already gentle: no change, no note
test_eq(flow([sol, sol, gap]), ('steady', ''))           # one miss is not a trend

In a session, two misses make the next question easier, and its "why" line says so. `Tutor.next` takes the pace from the session over the pace in the profile.

In [ ]:
fw = temp_learning(); fw.put('Ranking', 'Order items by a score.', 'ml')
fs = Session(Tutor(fw, fake2, context=False), limit=5)
for _ in range(2): fs.next(); fs.give_up()
test_eq(fs.next().why.endswith('; easier: you missed the last two'), True)
fw.put('Sandhi', 'Sounds change where words meet.', 'sanskrit')
test_eq(Tutor(fw, fake2, context=False).next(subject='sanskrit', pace='gentle').depth, 1)   # a new concept, at depth 1 and not 3

`retests` keeps a miss until a pass comes at least two questions later. A new miss moves the concept to the end.

In [ ]:
def lg(*xs): return [dict(concept=k, outcome=o, correct=o not in MISSES) for k, o in xs]
test_eq(retests(lg(('a', 'gap'), ('a', 'solid'))), {'a': 0})                          # a pass right after: still to retest
test_eq(retests(lg(('a', 'gap'), ('b', 'solid'), ('b', 'solid'), ('a', 'fragile'))), {})   # a later pass clears it
test_eq(retests(lg(('a', 'gap'), ('b', 'misconception'), ('a', 'gap'))), {'b': 1, 'a': 2})

When the questions left are only enough for the retests, the session asks the missed concept again, and says why. With more questions left, it does not.

In [ ]:
rl = temp_learning(); rl.put('Ranking', 'Order items by a score.', 'ml', order=0); rl.put('Sandhi', 'Sounds change where words meet.', 'sanskrit', order=1)
rs = Session(Tutor(rl, fake2, context=False), limit=4)
rs.log = lg(('ranking', 'gap'), ('sandhi', 'solid'), ('sandhi', 'solid'))
q = rs.next()
test_eq((q.concept, q.why), ('ranking', 'a retest: you missed this earlier in this session'))
rs.tutor.open.clear(); rs.limit = 10
assert not rs.next().why.startswith('a retest')

`bridges` takes known concepts only, from other subjects first. A transfer question gets them in its prompt; an explain question does not.

In [ ]:
bl = temp_learning()
bl.put('Attention', 'Weights from query-key dot products.', 'ml', related=['softmax'])
for n, s, f in (('Softmax', 'ml', 3), ('Ranking', 'ml', 4), ('Sandhi', 'sanskrit', 3), ('Nakshatra', 'jyotisha', 1)): bl.put(n, f'{n}.', s); bl.update(n.lower(), floor=f)
test_eq(bridges(bl.concept('attention'), bl.concepts()), ['Sandhi', 'Softmax', 'Ranking'])
fb = Fake(dflt=dict(question='Where else?', answer='', rubric=[]))
tb = Tutor(bl, fb, context=False)
tb.ask_on(bl.concept('attention'), 5); assert 'Sandhi' in fb.calls[-1][0]
tb.ask_on(bl.concept('attention'), 3); assert 'Sandhi' not in fb.calls[-1][0]

After each `observe_every` closed questions, the profile learns in the background.

In [ ]:
class Now:
    "A pool that runs a job at once, for the test."
    def submit(self, f, *a): f(*a)
ol = temp_learning(); ol.put('Ranking', 'Order items by a score.', 'ml')
fo = Fake({**fake2.rules, 'You keep a short profile': dict(changes=[dict(field='confuses', add='ranks against scores', why='You missed it twice.')])}, dflt=fake2.dflt)
ot = Tutor(ol, fo, context=False, pool=Now(), observe_every=2)
for _ in range(2): ot.give_up(ot.next().id)
test_eq(ol.profile().confuses, ['ranks against scores'])

The sources of the learner ground the question, the grade and the lesson. First, import `passage`, which makes one passage of a source.

In [ ]:
from abhyasa.ground import passage

A source of notes grounds the question. The learner sees the citations only when the question closes.

In [ ]:
class Notes:
    name = 'notes'
    def search(self, q, k=4): return L([passage('vault:notes', 'n1', 'My ranking notes', 'Ranking orders items; RRF adds 1/(k + rank).')])
gl = temp_learning(); gl.put('Ranking', 'Order items by a score.', 'ml')
gt = Tutor(gl, fake2, sources=[Notes()])
q = gt.next()
assert 'My ranking notes' in fake2.calls[-1][0]                 # the question prompt has the passage
assert 'grounds' not in q and 'ctx' not in q                  # the learner does not see it yet
r = gt.answer(q.id, 'the rank position', confidence=80)
test_eq(r.grounds, [dict(src='vault:notes', ref='n1', title='My ranking notes', url='')])
assert 'My ranking notes' in fake2.calls[-1][0]                 # the grade prompt has it too

A function in `CLOSE_HOOKS` sees each closed question and can add to the result. The result names its concept.

In [ ]:
seen = []
def spy(t, q, r): seen.append(q.concept); r.extra = 1
CLOSE_HOOKS.append(spy)
try: r = gt.answer(gt.next().id, 'the rank position', confidence=80)
finally: CLOSE_HOOKS.remove(spy)
test_eq((seen, r.extra, r.concept), (['ranking'], 1, 'ranking'))

A follow-up is only what the learner says this time. The grade prompt has the earlier turns in order, then the follow-up.
A pass on the third message, after two nudges, is `fragile`.

In [ ]:
tl = temp_learning(); tl.put('Ranking', 'Order items by a score.', 'ml')
tt = Tutor(tl, fake, context=False)
q = tt.next()
tt.answer(q.id, 'sum the scores', confidence=60)
tt.answer(q.id, 'scale them first', confidence=60)
r = tt.answer(q.id, 'use the rank position', confidence=70)
p = next(c[0] for c in reversed(fake.calls) if 'You grade' in c[1])
assert p.index('Tutor (question)') < p.index('Learner: sum the scores') < p.index('Tutor (nudge)') < p.index('Learner: scale them first')
test_eq(p.split("Learner's answer: ")[1].split('\n')[0], 'use the rank position')   # only the follow-up
assert 'scale them first' not in p.split("Learner's answer")[1]
test_eq((r.done, r.correct, r.outcome), (True, True, 'fragile'))

## Teach on request

After a shaky pass, a lesson comes only when you ask. It leaves the learner state as the close left it, and a second ask gives the same lesson.

In [ ]:
sl = temp_learning(); sl.put('Ranking', 'Order items by a score.', 'ml')
ss = Session(Tutor(sl, fake, context=False), limit=3)
q = ss.next()
for a in ('sum the scores', 'scale them first', 'use the rank position'): r = ss.answer(a, confidence=70)
test_eq((r.done, r.outcome), (True, 'fragile'))
before = dict(sl.concept('ranking'))

The lesson builds on the first error, and the concept does not change.

In [ ]:
lesson = ss.teach(q.id)
assert 'Check:' in lesson and 'could not' not in lesson
test_eq({k: v for k, v in sl.concept('ranking').items()}, before)
test_eq(ss.teach(q.id), lesson)
test_eq(len([x for x in ss.tutor.shut[q.id].turns if x.get('kind') == 'lesson']), 1)

The session log keeps the lesson, so a restart shows it.

In [ ]:
test_eq(sl.session(ss.id).log[-1]['turns'][-1], dict(who='tutor', kind='lesson', text=lesson, method=ss.tutor.shut[q.id].method))

The grading prompt tells the model to explain a term that you say you do not know, and to read past speech-to-text noise.

In [ ]:
assert "I don't know what the loop means" in GRADE_SP and 'speech to text' in GRADE_SP

A right equation meets a rubric point, even when the words of the method are missing. A right answer gets a grade of 7 or more. The grading prompt says so.

In [ ]:
for s in ('contains or implies it', 'need not say each step in words', 'only the method or framing', '7-10', 'When `correct` is false, `grade` is 6 or less.', 'A right result from wrong reasoning is not correct.'):
    assert s in GRADE_SP, s

The question prompt asks for rubric points that are ideas. A point is never a required word or a step of a named method.

In [ ]:
assert 'Never make a rubric point a required word' in question_prompt(lr.concept('ranking'), 3)

## Best try counts

A try with a grade of `NEAR` or more shows that the learner has the idea. If the question later closes as a miss, it closes as `fragile` instead.
The lesson still shows when the tutor teaches. The concept keeps its floor and its ceiling. A model that grades each try with a set grade plays the tutor here.

In [ ]:
test_eq(NEAR, 7)
def model(*gs):
    "A fake model whose grader gives the grades `gs` in turn, all as not correct."
    it = iter(gs)
    bad = lambda p, sp: dict(correct=False, grade=next(it), first_error='You do not name the method.', misconception='', feedback='Say it in words.')
    return Fake({**fake.rules, 'You grade one answer': bad})
def miss_after(*gs, give_up=True):
    "Try once for each grade in `gs`, then ask for the lesson. The result and the concept, with a floor of 2 before."
    l = temp_learning(); l.put('Ranking', 'Order items by a score.', 'ml'); l.update('ranking', floor=2)
    bt = Tutor(l, model(*gs), context=False); q = bt.next()
    for g in gs[:2 if give_up else 3]: r = bt.answer(q.id, 'a try', confidence=40)
    return (bt.give_up(q.id) if give_up else r), l.concept('ranking'), q

Each try keeps its grade. A try graded 8 is the best try, so the lesson still shows, but the outcome is `fragile` and the floor and the ceiling stay.

In [ ]:
r, c, q = miss_after(8)
test_eq([x['grade'] for x in q.tries], [8])
test_eq((r.done, r.correct, r.outcome), (True, True, 'fragile'))
assert 'Check:' in r.lesson
test_eq((r.feedback, r.first_error, r.misconception), ('', '', ''))   # no wrong-reply nudge under the pass
test_eq((c.floor, c.ceiling, c.misses), (2, 6, 0))

A best try of 3 is no near pass. The question still closes as a gap, and the ceiling falls.

In [ ]:
r, c, q = miss_after(3)
test_eq((r.correct, r.outcome), (False, 'gap'))
test_eq((c.floor, c.ceiling, c.misses), (2, 4, 1))

The third wrong try counts the same way. A grade of 8 in an earlier try saves it. Grades of 3, 3 and 3 do not.

In [ ]:
r, c, q = miss_after(3, 8, 3, give_up=False)
test_eq((r.done, r.outcome), (True, 'fragile'))
test_eq([x['grade'] for x in q.tries], [3, 8, 3])
test_eq((r.feedback, r.first_error), ('', ''))   # the third reply was wrong: its nudge does not show
r, c, q = miss_after(3, 3, 3, give_up=False)
test_eq((r.done, r.outcome), (True, 'gap'))

A try with no grade is no near pass.

In [ ]:
r, c, q = miss_after(None)
test_eq((r.correct, r.outcome), (False, 'gap'))

## Ways to learn, and gaps you name

Each question gets a method. The prompt says it, and the outcome counts for it when the question closes.

In [ ]:
import random
wl = temp_learning(); wl.put('Ranking', 'Order items by a score.', 'ml')
seen = []
def wgrade(p, sp):
    seen.append(p); said = p.split("Learner's answer:")[1].split("Learner's think-aloud:")[0]
    if 'the loop' in said: return dict(correct=False, grade=2, first_error='You skip the sum.', misconception='', feedback='A loop adds each product.',
                                    gaps=['weighted sum as a loop'], asks_for='code')
    return dict(correct='rank position' in said, grade=8, first_error='', misconception='', feedback='Yes.', gaps=[], asks_for='')
wfake = Fake({'You grade one answer': wgrade, 'The idea: weighted sum as a loop': dict(name='Weighted sum as a loop', statement='A loop adds each weight times its input.'),
              'Teach the smallest step': 'Positions are shared. Check: what if k = 0?',
              'Write ONE question': lambda p, sp: seen.append(p) or dict(question='Why do ranks merge well?', answer='They are comparable.', rubric=['comparable'])})
wt = Tutor(wl, wfake, context=False); wt.rng = random.Random(1)
q = wt.next()
assert q.method in METHODS and f"The way to teach this time: {METHODS[q.method]}" in seen[-1]

"I don't know what the loop means" keeps the idea as a gap of the concept, and "show me with code" switches the method for the next nudge.

In [ ]:
r = wt.answer(q.id, "I don't know what the loop means, show me with code")
test_eq(wl.concept('ranking').gaps, ['weighted-sum-as-a-loop'])
test_eq(wt.open[q.id].method, 'code')
test_eq(wl.attempts('ranking')[0].gaps, ['weighted sum as a loop'])
r = wt.answer(q.id, 'use the rank position')
assert METHODS['code'] in seen[-1]

The close counts the outcome for the method. The gap now comes first on the concept.

In [ ]:
test_eq(wl.stats().methods['ml']['code'], [.5, .5])   # a pass after a wrong try is shaky: half a credit
test_eq(wt.next(focus='ranking').concept, 'weighted-sum-as-a-loop')

"Another way" writes a new lesson in a method not used yet on that question, and records the request on the attempt.

In [ ]:
l1 = wt.teach(q.id); l2 = wt.teach(q.id, another=True)
ms = [x['method'] for x in wt.shut[q.id].turns if x.get('kind') == 'lesson']
test_eq((len(ms), ms[0] != ms[1]), (2, True))
test_eq(wl.attempts('ranking')[0].taught, 2)
test_eq(wl.stats().methods['ml']['code'], [.5, .5])   # a lesson on request changes no counts

A drill session takes the weak concept first, says why, and keeps the drill after a restart.

In [ ]:
dl = temp_learning(); dl.put('Ranking', 'Order items by a score.', 'ml'); dl.put('Sandhi', 'Sounds change where words meet.', 'sanskrit')
dl.update('sandhi', misses=2, misconceptions=['vowels never change'], seen=2)
ds = Session(Tutor(dl, fake, context=False), drill=True, limit=4)
q = ds.next()
test_eq((q.concept, q.why.split(';')[0]), ('sandhi', 'a weak spot: a misconception: vowels never change'))
test_eq(Session.load(ds.tutor, ds.id).drill, True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()